# Problem Solving 107: Fix the Analyst's Broken Queries
A new intern at Nkwen Traders sent five results to the operations manager. **Every query has a bug.**
One gives an error. The others run without any error but give a wrong answer, which is more dangerous.

For each query:
1. Run it and describe the **symptom** (what looks wrong?).
2. Explain the **cause** in one or two sentences.
3. Write the **fixed query** and run it.
4. Write **one sanity check** that would have caught the bug.

**What to submit:** the GitHub commit link to this notebook, and in the text box a short table: bug number, cause, fix, correct result.

In [ ]:
import sqlite3
import pandas as pd

SALES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch2_sales.csv"
BRANCHES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_branches.csv"

# 1. Load the two CSV files into pandas (Colab downloads them, not your phone or laptop)
sales = pd.read_csv(SALES_URL)
branches = pd.read_csv(BRANCHES_URL)

# 2. Create a small SQLite database file and copy both DataFrames into it as tables
conn = sqlite3.connect("nkwen.db")
sales.to_sql("sales", conn, if_exists="replace", index=False)
branches.to_sql("branches", conn, if_exists="replace", index=False)

# 3. A helper: run a SQL query and get the result back as a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite version:", sqlite3.sqlite_version)
print("Tables:", run_sql("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

## Bug 1: "There were no sales in March"
The intern says the till system must have been down all of March.

In [ ]:
broken_1 = """
SELECT COUNT(*) AS transactions, SUM(TotalSale_FCFA) AS revenue
FROM sales
WHERE strftime('%m', Date) = 3
"""
run_sql(broken_1)

**Symptom (TODO):**

**Cause (TODO):** hint - run `run_sql("SELECT strftime('%m', '2026-03-15') AS m, typeof(strftime('%m', '2026-03-15')) AS t")`.

In [ ]:
fixed_1 = """
-- TODO: fixed query
"""
f1 = run_sql(fixed_1)
f1

In [ ]:
# Self-check 1
assert f1["transactions"][0] > 0, "Your fixed query still finds no March sales."
assert f1["transactions"][0] < len(sales), "Your fixed query returns every sale, not only March."
print("Success: March sales found.")

**Sanity check that would have caught it (TODO):**

## Bug 2: "Nkwen's share of total revenue is 0%"
`CASE WHEN ... THEN ... ELSE ... END` works like an if/else inside a query: here it keeps Nkwen's sale value and puts 0 for other branches.

In [ ]:
broken_2 = """
SELECT SUM(CASE WHEN Branch = 'Nkwen' THEN TotalSale_FCFA ELSE 0 END)
       / SUM(TotalSale_FCFA) * 100 AS nkwen_share_pct
FROM sales
"""
run_sql(broken_2)

**Symptom (TODO):**

**Cause (TODO):** hint - compare `run_sql("SELECT 7 / 2 AS a, 7.0 / 2 AS b")`.

In [ ]:
fixed_2 = """
-- TODO: fixed query (round the share to 1 decimal place)
"""
f2 = run_sql(fixed_2)
f2

In [ ]:
# Self-check 2
share = f2.iloc[0, 0]
assert share is not None and 0 < share < 100, "A branch's share should be between 0 and 100."
print("Success: the share is a real percentage now.")

**Sanity check that would have caught it (TODO):**

## Bug 3: "Limbe Down Beach has already made its first sale"
The new Limbe branch is planned for 2027.

In [ ]:
broken_3 = """
SELECT b.Branch, COUNT(*) AS transactions
FROM branches AS b
LEFT JOIN sales AS s ON s.Branch = b.Branch
GROUP BY b.Branch
ORDER BY transactions
"""
run_sql(broken_3)

**Symptom (TODO):**

**Cause (TODO):**

In [ ]:
fixed_3 = """
-- TODO: fixed query
"""
f3 = run_sql(fixed_3)
f3

In [ ]:
# Self-check 3: the transactions should add up to the real number of sales
assert len(f3) == len(branches), "Keep one row per branch in the branch table."
assert f3["transactions"].sum() == len(sales), "Your transactions do not add up to the number of rows in sales. Which rows are you still counting?"
print("Success: every branch is listed and the counts add up to the real number of sales.")

**Sanity check that would have caught it (TODO):**

## Bug 4: "Which branches made more than 15 million FCFA?"
This one gives an error.

In [ ]:
broken_4 = """
SELECT Branch, SUM(TotalSale_FCFA) AS revenue
FROM sales
WHERE SUM(TotalSale_FCFA) > 15000000
GROUP BY Branch
"""
try:
    display(run_sql(broken_4))
except Exception as error:
    print("Error:", error)

**Symptom (TODO):** copy the important part of the error message.

**Cause (TODO):**

In [ ]:
fixed_4 = """
-- TODO: fixed query (highest revenue first)
"""
f4 = run_sql(fixed_4)
f4

**Sanity check that would have caught it (TODO):**

## Bug 5: "Mobile money sales at Nkwen"
The manager asked: how many Nkwen sales were paid with Mobile Money or Orange Money?

In [ ]:
broken_5 = """
SELECT COUNT(*) AS transactions, SUM(TotalSale_FCFA) AS revenue
FROM sales
WHERE PaymentMethod = 'Mobile Money' OR PaymentMethod = 'Orange Money' AND Branch = 'Nkwen'
"""
run_sql(broken_5)

**Symptom (TODO):** hint - compare the result with the total number of Nkwen transactions.

**Cause (TODO):**

In [ ]:
fixed_5 = """
-- TODO: fixed query
"""
f5 = run_sql(fixed_5)
f5

In [ ]:
# Self-check 4 (bugs 4 and 5)
assert len(f4) > 0, "Bug 4: your query returns no branches."
assert (f4["revenue"] > 15_000_000).all(), "Bug 4: every branch in the result should be above 15,000,000 FCFA."
nkwen_total = run_sql("SELECT COUNT(*) AS n FROM sales WHERE Branch = 'Nkwen'")["n"][0]
assert 0 < f5["transactions"][0] < nkwen_total, "Bug 5: the answer must be a part of Nkwen's transactions, not more."
print("Success: Bug 4 filters the groups, and Bug 5 is a subset of Nkwen's transactions.")

**Sanity check that would have caught it (TODO):**

## Your checklist (TODO)
Write 5 short checks you will run **before you send any query result to a manager**.
For example: "Do the parts add up to the total?"
1.
2.
3.
4.
5.

### Before you submit
- [ ] **Runtime > Restart session and run all**: every cell runs from top to bottom without an error.
- [ ] Every self-check cell printed its success message.
- [ ] Your answers are written in Markdown cells in your own words, not only as code output.
- [ ] Save to GitHub: **File > Save a copy in GitHub**, choose your repository, write a clear commit message, and click OK.
- [ ] Open the notebook on GitHub, check that the outputs show, and copy the commit link for your submission.